## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [13]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
from mlflow.models import infer_signature
...

Ellipsis

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [ ]:
# Implement transformation logic as in session 2
import joblib

df_input = pd.read_csv("../Session2_class/datasets/Churn_Modelling_train_test.csv")

def balance_dataset(df: pd.DataFrame, target: str) -> pd.DataFrame:
    df_y0 = df[df[target] == 0]
    df_y1 = df[df[target] == 1]
    
    # Undersampling the majority class
    min_size = len(df_y0)
    df_majority_downsampled = df_y0.sample(n=min_size, random_state=42)
    
    df = pd.concat([df_majority_downsampled, df_y0])
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced = balance_dataset(df_input, 'Exited')

class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = ['RowNumber', 'CustomerId', 'Surname']
        self.CATEGORICAL_COLUMNS = ['Geography', 'Gender']
        self.encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")

    def transform(self, df: pd.DataFrame, is_training: bool = True) -> pd.DataFrame:
        df = df.drop(columns=self.DROP_COLUMNS)

        self.encoder.fit(df[self.CATEGORICAL_COLUMNS])
            
        encoded_features = self.encoder.transform(df[self.CATEGORICAL_COLUMNS])
        df = df.drop(columns=self.CATEGORICAL_COLUMNS)
        df = pd.concat([df, encoded_features], axis=1)
        
        return df
transformer = Transformer()
df_transformed = transformer.transform(df_balanced)

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = df_transformed.drop('Exited', axis=1)
y = df_transformed['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "max_depth": 6,
    "min_samples_split": 10,
    "random_state": 21
}

dt_model = DecisionTreeClassifier(**params)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 1.0000
F1 Score: 0.0000


c:\Users\sudhi\Downloads\MLOps\Session 1_ML\mlops-and-system-design\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("Practice Experiment S3- Hemangini Jena")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Session3 Assignment", "Bank Customer Churn ML Experiment")

    # Infer the model signature
    signature = infer_signature(X_train, dt_model.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=dt_model,
        artifact_path="decision_tree_bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="churn-model-dt")

c:\Users\sudhi\Downloads\MLOps\Session 1_ML\mlops-and-system-design\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'churn-model-dt' already exists. Creating a new version of this model...
2026/05/24 12:15:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model v

🏃 View run upbeat-lamb-247 at: http://127.0.0.1:8080/#/experiments/658896502001904195/runs/5a051008fbe546f5bd02b6e4c30e132d
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/658896502001904195


Created version '3' of model 'churn-model-dt'.


In [ ]:
import joblib

PATH = 'model_utils/'
joblib.dump(Transformer().transform.encoder, 'encoder.pkl')
joblib.dump(dt_model, f'{PATH}dt_model.pkl')

['model_utils/dt_model.pkl']

In [21]:
model_uri='runs:/5a051008fbe546f5bd02b6e4c30e132d/bank_model'

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [24]:
# import validation dataset to test inference
df_validation = pd.read_csv("../Session3_class/val_datasets/Churn_Modelling_val.csv")
print(df_validation.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender   Age  \
0       8607    15694581  Rawlings          807     Spain    Male  42.0   
1       4685    15736963   Herring          623    France    Male  43.0   
2       1732    15721730    Amechi          601     Spain  Female  44.0   
3       4743    15762134     Liang          506   Germany    Male  59.0   
4       4522    15648898    Chuang          560     Spain  Female  27.0   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       5       0.00              2        1.0             1.0   
1       1       0.00              2        1.0             1.0   
2       4       0.00              2        1.0             0.0   
3       8  119152.10              2        1.0             1.0   
4       7  124995.98              1        1.0             1.0   

   EstimatedSalary  Exited  
0         74900.90       0  
1        146379.30       0  
2         58561.31       0  
3        170679.74       0  
4      

In [25]:
y_validation = df_validation["Exited"]
df_validation = df_validation.drop("Exited", axis = 1)

In [23]:
import requests
import json

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [29]:
# transform data - if necessary
import joblib

class ModelBatchPredictor:
    CATEGORICAL_COLUMNS = ['Geography', 'Gender']
        
    def __init__(self):
        self.encoder = joblib.load(f'{PATH}encoder.pkl')
        self.transformer = Transformer()

    def _apply_transformation(self, cls, df: pd.DataFrame):
        df = self.transformer.transform(df.copy(), is_training=False)
        encoded_df = self.encoder.transform(df[cls.CATEGORICAL_COLUMNS])
        df = df.drop(columns=cls.CATEGORICAL_COLUMNS)
        df = pd.concat([df, encoded_df], axis=1)

        return df 

##### Batch Inference

In [30]:
# define a function to implement batch inference with mlflow
def batch_inference_with_mlflow(model_uri: str, input: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)
    input = ModelBatchPredictor.apply_transformation(input)
    predictions = model.predict(input)
        
    return predictions
    
def batch_inference_without_mlflow(model_path: str, input: pd.DataFrame):
    model = joblib.load(model_path)
    input = ModelBatchPredictor._apply_transformation(input)
    predictions = model.predict(input)

    return predictions

In [ ]:
# define the model uri that should be used
model_uri = 'runs:/5a051008fbe546f5bd02b6e4c30e132d/bank_model'
input = df_validation
batch_prediction_result = batch_inference_with_mlflow(model_uri, input)

MlflowException: The following failures occurred while downloading one or more artifacts from http://127.0.0.1:8080/api/2.0/mlflow-artifacts/artifacts/658896502001904195/5a051008fbe546f5bd02b6e4c30e132d/artifacts:
##### File bank_model #####
API request to http://127.0.0.1:8080/api/2.0/mlflow-artifacts/artifacts/658896502001904195/5a051008fbe546f5bd02b6e4c30e132d/artifacts/bank_model failed with exception HTTPConnectionPool(host='127.0.0.1', port=8080): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/658896502001904195/5a051008fbe546f5bd02b6e4c30e132d/artifacts/bank_model (Caused by ResponseError('too many 500 error responses'))

In [ ]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix

print(batch_prediction_result[:5])
print("----")
print(confusion_matrix(y_validation, batch_prediction_result))
print("----")
print(accuracy_score(y_validation, batch_prediction_result))

In [ ]:
batch_prediction_without_mlflow = ModelBatchPredictor().batch_inference_without_mlflow(
    model_path = f'{PATH}dt_model.pkl'
    input = df_validation
)

In [ ]:
print(batch_prediction_without_mlflow[:5])
print("----")
print(confusion_matrix(y_validation, batch_prediction_without_mlflow))
print("----")
print(accuracy_score(y_validation, batch_prediction_without_mlflow))

##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [ ]:
import requests
import json

In [ ]:
# import validation dataset to test inference - just one record
df_validation = pd.read_csv("../Session3_class/val_datasets/Churn_Modelling_val.csv").head(1)
print(df_validation["Exited"])
df_validation = df_validation.drop("Exited", axis = 1)

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [ ]:
# transform data - if necessary
class MLflowInferenceOnline:
    ONE_HOT_ENCODE_COLUMNS= [
            "Geography",
            "Gender",
        ]
    def __init__(self, host="http://127.0.0.1", port=5000):
        self.url = f"{host}:{port}/invocations"
        self.encoder = joblib.load(f'{PATH}encoder.pkl')
        self.transformer = Transformer()

    def _apply_transformation(self,cls, df: pd.DataFrame):
        df = self.transformer.transform(df.copy(), df)
        encoded_df = self.encoder.transform(df[cls.ONE_HOT_ENCODE_COLUMNS])
        df = df.drop(columns=cls.ONE_HOT_ENCODE_COLUMNS)
        df = pd.concat([df, encoded_df], axis=1)

        return df 

    def predict_csv(self, csv_data: str):
        """Send CSV-formatted request"""
        headers = {"Content-Type": "text/csv"}
        response = requests.post(self.url, headers=headers, data=csv_data)
        return response
    
    
ml_flow_online_inference_client = MLflowInferenceOnline()

In [ ]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(pandas_data: pd.DataFrame):
        headers = {"Content-Type": "application/json"}
        pandas_data = MLflowInferenceOnline._apply_transformation(pandas_data)
        payload = {"dataframe_split": pandas_data.to_dict(orient="split")}
        response = requests.post(MLflowInferenceOnline.url, headers=headers, data=json.dumps(payload))
        return response

ml_flow_online_inference_client = MLflowInferenceOnline()

In [ ]:
response_pandas = online_inference_pandas(df_validation)
response_pandas.content

In [ ]:
json_data = {
    "dataframe_split": { 
    "columns": ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary", "Geography_Germany", "Geography_Spain", "Geography_nan", "Gender_Male"],
    "data": [[634, 27.0, 3, 107027.52, 1, 1.0, 0.0, 173425.68, 1.0, 0.0, 0.0, 1.0]]}}

In [ ]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(json_data: dict):
    """Send JSON (pandas-style) request"""
    headers = {"Content-Type": "application/json"}
    response = requests.post(MLflowInferenceOnline.url, headers=headers, json=json_data)
    return response

In [ ]:
response_json = online_inference_json(json_data)
response_json.content